<a href="https://colab.research.google.com/github/ranggasaviory/Tugas-Praktikum-1-AI-Rangga/blob/main/prediksi_penyakit_jantung_apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install streamlit pyngrok pandas scikit-learn --quiet

In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

@st.cache_data
def load_data():
    # Menggunakan dataset yang stabil
    url = "https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv"
    df = pd.read_csv(url)

    # Bersihkan data kotor (?) dan ubah ke angka
    df = df.replace('?', np.nan).dropna().apply(pd.to_numeric)

    # Pastikan ada kolom target (biasanya kolom terakhir)
    if 'target' not in df.columns:
        df.rename(columns={df.columns[-1]: 'target'}, inplace=True)

    # Binerkan target (0 = sehat, 1 = risiko)
    df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)
    return df

def main():
    st.set_page_config(page_title="Heart Predictor", layout="wide")
    st.title("❤️ Prediksi Risiko Penyakit Jantung")

    try:
        df = load_data()
        X = df.drop("target", axis=1)
        y = df["target"]

        # Training model sederhana
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X, y)

        st.sidebar.header("Input Data Pasien")

        # Cara Aman: Ambil data berdasarkan urutan index kolom agar tidak error nama
        # Kolom 0: Age, Kolom 3: Trestbps, Kolom 4: Chol, Kolom 7: Thalach
        age = st.sidebar.slider("Umur", int(df.iloc[:, 0].min()), int(df.iloc[:, 0].max()), 45)
        sex = st.sidebar.selectbox("Kelamin (1=Pria, 0=Wanita)", [1, 0])
        trestbps = st.sidebar.slider("Tekanan Darah Darah", int(df.iloc[:, 3].min()), int(df.iloc[:, 3].max()), 120)
        chol = st.sidebar.slider("Kolesterol", int(df.iloc[:, 4].min()), int(df.iloc[:, 4].max()), 200)

        if st.button("Cek Hasil Prediksi"):
            # Buat template input berdasarkan rata-rata (mean) data asli
            input_raw = X.mean().values.copy()

            # Masukkan nilai dari slider ke posisi kolom yang sesuai
            input_raw[0] = age
            input_raw[1] = sex
            input_raw[3] = trestbps
            input_raw[4] = chol

            prediction = model.predict([input_raw])[0]

            st.write("---")
            if prediction == 1:
                st.error("### ⚠️ Hasil: Berisiko Penyakit Jantung")
                st.write("Disarankan untuk berkonsultasi dengan dokter ahli jantung.")
            else:
                st.success("### ✅ Hasil: Risiko Rendah")
                st.write("Tetap jaga pola makan dan olahraga rutin!")

    except Exception as e:
        st.error(f"Kesalahan Sistem: {e}")
        st.write("Coba jalankan ulang cell deployment di Colab.")

if __name__ == "__main__":
    main()

Overwriting app.py


In [26]:
import time
from pyngrok import ngrok

# Bersihkan sisa proses
!pkill streamlit
ngrok.kill()

# Gunakan Token Anda
AUTH_TOKEN = "3DHsRk8nFHjtvhQ1jgjRxkR2H0o_3MTkidfx5kJpyo8RiCqMp"
ngrok.set_auth_token(AUTH_TOKEN)

# Jalankan
get_ipython().system_raw('streamlit run app.py &')
print("Menghubungkan ke server (10 detik)...")
time.sleep(10)

try:
    url = ngrok.connect(8501).public_url
    print(f"\n✅ SIAP DIGUNAKAN!")
    print(f"🔗 Klik Link Ini: {url}")
except Exception as e:
    print(f"❌ Gagal koneksi: {e}")

Menghubungkan ke server (10 detik)...

✅ SIAP DIGUNAKAN!
🔗 Klik Link Ini: https://reconvene-slimness-sensitize.ngrok-free.dev
